# Symbolic verification of the Prox-TMM Lyapunov simplification

This notebook verifies the algebraic identity used in the Prox-TMM Lyapunov proof:

$$
\mathcal S_k^\infty=0.
$$

The notebook is organized as follows:

1. define symbols matching the notation used in the paper;
2. apply the update-rule substitutions;
3. transcribe $\mathcal S_k^\infty$ in those symbols;
4. let SymPy reduce the resulting expression to zero.

We work in one scalar coordinate. Since the slack is a finite linear combination of inner products and squared norms with scalar coefficients, this scalar identity proves the Hilbert-space identity coordinatewise.

## 1. Symbols matching the proof

| Notation | Python identifier |
|---|---|
| $\sqrt q$, $q$, $L$, $\mu$ | `sqrt_q`, `q`, `L`, `mu` |
| $\theta=(1-\sqrt q)^2$ | `theta` |
| $y^{k-1}$, $y^k$ | `y_km1`, `y_k` |
| $x^\star$ | `x_star` |
| $z^k$, $z^{k+1}$ | `z_k`, `z_kp1` |
| $s^k$, $s^{k+1}$, $s^\star$ | `s_k`, `s_kp1`, `s_star` |
| $\nabla f(y^k)$, $\nabla f(y^{k-1})$, $\nabla f(x^\star)$ | `nabla_f_y_k`, `nabla_f_y_km1`, `nabla_f_x_star` |
| $\langle u,v\rangle$, $\|v\|^2$ | `inner(u, v)`, `norm2(v)` |

In [ ]:
import sympy as sp

sqrt_q, L = sp.symbols("sqrt_q L", nonzero=True)
y_km1, y_k, x_star, z_k, z_kp1 = sp.symbols("y_km1 y_k x_star z_k z_kp1")
s_k, s_kp1, s_star = sp.symbols("s_k s_kp1 s_star")

q = sqrt_q**2
mu = q * L
theta = (1 - sqrt_q)**2

inner = lambda u, v: u * v
norm2 = lambda v: v**2

## 2. Substitutions used by SymPy

We now apply the update rules and definitions used in the paper. This gives the substitutions

$$
q=(\sqrt q)^2,
\qquad
\mu=qL,
\qquad
\theta=(1-\sqrt q)^2,
$$

$$
\nabla f(x^\star)=-s^\star,
$$

$$
\nabla f(y^k)
=\sqrt q\,L\big((1-\sqrt q)z^k+\sqrt q\,y^k-z^{k+1}\big)-s^{k+1},
$$

$$
x^k=\frac{(1+\sqrt q)y^k-2\sqrt q\,z^k}{1-\sqrt q},
\qquad
\nabla f(y^{k-1})=L(y^{k-1}-x^k)-s^k.
$$

The scalar substitutions for $q$, $\mu$, $\theta$, and $\nabla f(x^\star)$ are copied directly from initialization, shorthand, and definition. For the gradient substitutions, start with the $\bar z$-update and sampled-subgradient,

$$
\bar z^{k+1}
=(1-\sqrt q)z^k+\sqrt q\,y^k
-\frac{1}{\sqrt q L}\nabla f(y^k),
\qquad
s^{k+1}=\sqrt q\,L(\bar z^{k+1}-z^{k+1}).
$$

The sampled-subgradient gives $\bar z^{k+1}=z^{k+1}+(\sqrt q L)^{-1}s^{k+1}$. Substituting this into the $\bar z$-update gives

$$
z^{k+1}+\frac{1}{\sqrt q L}s^{k+1}
=(1-\sqrt q)z^k+\sqrt q\,y^k
-\frac{1}{\sqrt q L}\nabla f(y^k).
$$

Solving for $\nabla f(y^k)$ gives

$$
\nabla f(y^k)
=\sqrt q\,L\big((1-\sqrt q)z^k+\sqrt q\,y^k-z^{k+1}\big)-s^{k+1}.
$$

For the previous-step gradient, use the $x$-update at the previous index and the sampled-subgradient definition,

$$
x^k
=y^{k-1}-\frac{1}{L}\nabla f(y^{k-1})
-\sqrt q(\bar z^k-z^k),
\qquad
s^k=\sqrt q\,L(\bar z^k-z^k).
$$

These two give $x^k=y^{k-1}-L^{-1}(\nabla f(y^{k-1})+s^k)$, hence $\nabla f(y^{k-1})=L(y^{k-1}-x^k)-s^k$. To express this only in the variables used by SymPy, use the extrapolation relation

$$
y^k=\frac{(1-\sqrt q)x^k+2\sqrt q\,z^k}{1+\sqrt q}
$$

Solving for $x^k$ gives

$$
x^k=\frac{(1+\sqrt q)y^k-2\sqrt q\,z^k}{1-\sqrt q},
$$

which is the expression used in the code below.

In [ ]:
nabla_f_x_star = -s_star

nabla_f_y_k = sqrt_q * L * ((1 - sqrt_q) * z_k + sqrt_q * y_k - z_kp1) - s_kp1

x_k = ((1 + sqrt_q) * y_k - 2 * sqrt_q * z_k) / (1 - sqrt_q)
nabla_f_y_km1 = L * (y_km1 - x_k) - s_k

## 3. Slack expression

The following cell transcribes $\mathcal S_k^\infty$ after function-value cancellation, using the substitutions above.

In [ ]:
S_k_infty = 0
S_k_infty += (1 - theta) / (2 * L) * norm2(s_kp1 - s_star)
S_k_infty += theta / (2 * L) * norm2(s_k - s_kp1)

S_k_infty += -(1 - q) * (1 - theta) * (
    inner(nabla_f_y_k, x_star - y_k) + mu / 2 * norm2(x_star - y_k)
)
S_k_infty += -((1 - q) * (1 - theta)) / (2 * (L - mu)) * norm2(
    nabla_f_x_star - nabla_f_y_k - mu * (x_star - y_k)
)

S_k_infty += -(1 - q) * theta * (
    inner(nabla_f_y_k, y_km1 - y_k) + mu / 2 * norm2(y_km1 - y_k)
)
S_k_infty += -((1 - q) * theta) / (2 * (L - mu)) * norm2(
    nabla_f_y_km1 - nabla_f_y_k - mu * (y_km1 - y_k)
)

S_k_infty += -(1 - q) * (
    inner(nabla_f_x_star, y_k - x_star) + mu / 2 * norm2(y_k - x_star)
)
S_k_infty += -(1 - q) / (2 * (L - mu)) * norm2(
    nabla_f_y_k - nabla_f_x_star - mu * (y_k - x_star)
)

S_k_infty += (1 - q) * theta * (
    inner(nabla_f_x_star, y_km1 - x_star) + mu / 2 * norm2(y_km1 - x_star)
)
S_k_infty += ((1 - q) * theta) / (2 * (L - mu)) * norm2(
    nabla_f_y_km1 - nabla_f_x_star - mu * (y_km1 - x_star)
)

S_k_infty += -(1 + q - theta) * inner(s_kp1, x_star - z_kp1)
S_k_infty += q * theta * inner(s_k, x_star - z_k)
S_k_infty += -(1 + q - theta - sqrt_q * theta) * inner(s_star, z_kp1 - x_star)
S_k_infty += (q - sqrt_q) * theta * inner(s_star, z_k - x_star)
S_k_infty += -sqrt_q * theta * inner(s_k, z_kp1 - z_k)

S_k_infty += -theta / (2 * L) * norm2(s_k - s_star)
S_k_infty += 1 / (2 * L) * norm2(s_kp1 - s_star)
S_k_infty += mu * norm2(z_kp1 - x_star)
S_k_infty += -mu * theta * norm2(z_k - x_star)

## 4. SymPy reduction

After the substitutions, every term is a rational expression in the same scalar symbols. SymPy combines denominators and factors the result.

In [ ]:
simplified_slack = sp.factor(sp.together(sp.expand(S_k_infty)))
assert simplified_slack == 0
simplified_slack

The zero expression proves

$$
\mathcal S_k^\infty=0.
$$